# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第6章：正答率や有無を扱う：一般化線形混合モデル（GLMM）


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照論文: Jaeger (2008), *Journal of Memory and Language*／Idemaru & Holt (2011), *Journal of Experimental Psychology*。

In [ ]:
library(lme4)
library(lmerTest)

In [ ]:
# ---- 共通データセットの読み込み ----
# 第1章と全く同じ手順・同じ乱数seedでVOTデータを作り直す。
# 「本ごとに違うデータ」を避けるためだが、外部URLから読み込むのではなく
# その場で再生成する設計にしてある——理由は、配布用ノートブックが
# 非公開のGitHubリポジトリのURLに依存すると、読者の環境からは
# アクセスできず404になってしまうため（実際にこの問題が起きて修正した）。
# コードを自分の環境で動かす分には、この方式のほうが「ネットワーク不要で
# 必ず同じ結果になる」という利点もある。
if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5章で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5章以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、分散推定値がすべて0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5章で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3章のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

vot_data <- vot_data %>%
  mutate(
    # 文字列のままだとRが「順序のない値」として扱うため、
    # 話者・単語・性別・条件をすべてfactor（因子）型に変換しておく。
    # 特にconditionはlevels=でvoiced/voicelessの順序を明示し、
    # 以降のモデルの切片が「voiced群の平均」になるよう固定する。
    speaker   = factor(speaker),
    word      = factor(word),
    gender    = factor(gender),
    condition = factor(condition, levels = c("voiced", "voiceless"))
  )

glimpse(vot_data)  # 型と最初の数行を確認。数値であるべき列が<chr>のままなら要注意

# --- 実行結果 ---
# Rows: 200
# Columns: 6
# $ speaker   <fct> S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, ...
# $ word      <fct> word_1, word_2, word_3, word_4, word_5, word_6, word_7, ...
# $ gender    <fct> M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, ...
# $ condition <fct> voiced, voiceless, voiced, voiceless, voiced, voiceless, ...
# $ VOT       <dbl> 28.129829, 88.042064, 30.742652, 94.104833, 28.438207, ...
# $ F0        <dbl> 119.0617, 144.9710, 148.8772, 148.7330, 109.2904, ...
#
# 【出力の読み方】
# ・Rows: 200 / Columns: 6 が行数と列数。10話者×20語=200行になっているかを
#   まず確認する。ここが合っていなければ、以降の分析はすべて無意味になる。
# ・各行の <fct> <dbl> が**型**である。<fct>はfactor（水準を持つカテゴリ）、
#   <dbl>は小数を含む数値。speaker・word・gender・conditionがすべて<fct>、
#   VOTとF0が<dbl>になっていれば正しい。
#   **VOTが<chr>（文字列）になっていたら、そこで止まって読み込みを見直す。**
#   文字列のままでは平均もモデルも計算できず、しかもエラーではなく
#   「変な結果」として静かに進んでしまうことがある。
# ・conditionの並びがvoiced, voiceless, voiced, voiceless...と1行おきに
#   交互になっている。これは意図した設計どおり（どの話者も両条件を10回ずつ持つ）。
# ・speakerが先頭でS01ばかり並ぶのは、話者ごとに20行がまとまっているため。
#   行の並び順自体は分析結果に影響しない（第7章のAR(1)を使うときだけ例外）。


■ 二値応答変数を作る：

**データについて**: ここからは、第1章で作った音響測定値（VOT）そのものではなく、「聞き手がその発音を条件どおり（典型的）に知覚したかどうか」という**知覚判断**を擬似的にシミュレーションする。音響測定と知覚判断は別種のデータであり、後者は必ず0/1（あるいは選択肢のうちどれか）の離散値になる——これが本章で回帰やt検定ではなくGLMMが必要になる理由である。

**実際の研究例**：Idemaru & Holt (2011) は、VOTとF0という2つの音響手がかりが有声性の知覚判断（"pa" or "ba"）にどう重みづけされるかを強制選択課題で調べた。応答は各試行ごとに0/1で記録される——このような二値の試行データをどう分析すべきかが、本章のGLMMの主題である。

**どう改変したか**: 実験を新たに実施する代わりに、第1章のVOT値から「聞き手が典型的だと判断する確率」を計算し、その確率にもとづいて0/1をランダムに生成する（`rbinom()`）。判断は次の3つの要因の合成としてモデル化した。(1) 自分の条件の典型値からVOTがどれだけ離れているか（離れるほど非典型と判断されやすい）、(2) 条件そのものの効果（ここではvoicelessの方がやや非典型と判断されにくいという効果を仮に入れた）、(3) 話者ごとの判断の甘さ・厳しさのばらつき。

In [ ]:
set.seed(6)  # このセルだけの乱数固定（第1章のVOT生成とは別に、知覚判断の乱数を管理する）

vot_data <- vot_data %>%
  # group_by(condition) %>% mutate(...) %>% ungroup(): 「条件ごとに」計算してから
  # 元の1本のデータフレームに戻す、という意味。scale()で条件内の平均・SDを使って
  # 標準化（z得点）するために、条件ごとにグループ化してから計算する必要がある。
  group_by(condition) %>%
  mutate(vot_z = as.numeric(scale(VOT))) %>%  # 自分の条件内での標準化得点（0=典型的な中心、絶対値が大きいほど極端）
  ungroup() %>%
  mutate(
    # 話者ごとに固定の「判断の甘さ・厳しさ」を1つ割り当てる（正規分布からサンプル）。
    # as.numeric(factor(speaker))で話者ID(文字列)を1,2,3,...の整数に変換し、
    # 10人分の乱数ベクトルからその話者の値を選び出している。
    speaker_bias = rnorm(10, mean = 0, sd = 0.6)[as.numeric(factor(speaker))],
    # 単語ごとの「聞き取りやすさ」の差も同じ要領で入れる。これを入れないと
    # 後で当てはめる (1 | word) の分散がほぼ0になり、ランダム切片を置く意味が
    # データ側に存在しないことになる。
    word_bias = rnorm(20, mean = 0, sd = 0.5)[as.numeric(factor(word))],
    # ロジット（対数オッズ）の尺度で3つの効果を足し合わせる。
    #   1.3                          : 基準となる「典型と判断されやすさ」
    #   -0.8 * abs(vot_z)            : 条件の典型値から離れるほど非典型と判断されやすくなる
    #   voicelessなら-0.6            : 無声条件はやや非典型と判断されやすいと仮定
    #   speaker_bias                 : 話者ごとの個人差
    #   word_bias                    : 単語ごとの個人差
    logit_p = 1.3 - 0.8 * abs(vot_z) +
      if_else(condition == "voiceless", -0.6, 0) +
      speaker_bias + word_bias,
    p_typical = plogis(logit_p),               # ロジットを0〜1の確率に変換
    typical = rbinom(n(), size = 1, prob = p_typical)  # その確率で0/1を1回ずつ試行する
  )

table(vot_data$condition, vot_data$typical)  # 条件×典型性のクロス集計で確認

# --- 実行結果の例（実際にRで実行して確認済み）---
#            0  1
#   voiced    30 70
#   voiceless 47 53
# voiced条件はvoiceless条件より「典型的」と判断される行が多いはず
# （上の -0.6 という設計のとおり）。もし両方ともほぼ全部1（または全部0）に
# なっていたら、確率の作り方が偏りすぎている——plogis()に渡す値の範囲を見直すこと。


■ ロジスティック混合効果モデル：

Jaeger (2008) は、パーセント正答率をarcsine変換してANOVAに投入する伝統的手法が誤った第一種の過誤率をもたらすことを示し、ロジットのGLMMを推奨した。

In [ ]:
model_glmm <- glmer(
  # family = binomial により、応答を0/1（ベルヌーイ試行）として扱い、
  # 線形予測子をロジット関数でつなぐ（ロジスティック回帰の混合効果版）
  typical ~ condition + (1 | speaker) + (1 | word),
  family = binomial,
  data = vot_data
)

summary(model_glmm)
# 係数はログオッズの尺度。正なら「典型的」である確率が上がる方向。

# --- 実行結果（抜粋）---
# Random effects:
#  Groups  Name        Variance Std.Dev.
#  word    (Intercept) 0.1878   0.4333
#  speaker (Intercept) 0.3716   0.6096
# Number of obs: 200, groups:  word, 20; speaker, 10
#
# Fixed effects:
#                    Estimate Std. Error z value Pr(>|z|)
# (Intercept)          0.9518     0.3344   2.846  0.00442 **
# conditionvoiceless  -0.8136     0.3708  -2.194  0.02822 *
#
# 【出力の読み方】
# ・まず尺度を確認する。**係数はログオッズ（対数オッズ）であって、
#   確率でもミリ秒でもない。** 第5章までのように「52.5ms増える」と
#   読むことはできず、そのままでは大きさを実感できない。
#   だから次のセルで確率とオッズ比に変換する。
# ・(Intercept) 0.9518 は「voiced条件のログオッズ」。正の値なので、
#   voiced条件では『典型的』と判断されるほうが優勢だと分かる。
#   ログオッズが0なら確率50%、正なら50%超、負なら50%未満である。
# ・conditionvoiceless −0.8136 は「voicedからvoicelessに移るとログオッズが
#   0.81下がる」という意味。負なので**無声条件のほうが『典型的』と
#   判断されにくい**。第1章の生成コードで -0.6 を入れた効果に対応する
#   （推定値が -0.81 とやや大きいのは標本のばらつきの範囲内）。
# ・検定統計量が t ではなく **z** になっている点に注意する。線形混合効果
#   モデルと違い、GLMMでは残差の分散を別に推定しないため、
#   Satterthwaite近似のような自由度の調整が不要（かつ不可能）である。
#   したがって出力にdf列は無い。
# ・Random effects欄にResidual行が無いことも、線形モデルとの重要な違いである。
#   二項分布では分散が平均から決まる（平均pなら分散はp(1-p)）ので、
#   独立した残差分散というパラメータが存在しない。
# ・ランダム効果のSDもログオッズの尺度である。speaker 0.6096 は
#   「話者によって判断の甘さ・厳しさがログオッズで0.61ぶんばらつく」。
#   条件の効果（0.81）と同じくらいの大きさであり、**話者差を無視すれば
#   条件の効果を取り違えるだけの大きさがある**ことを意味する。
# ・word 0.4333 は単語ごとのばらつき。第1章で word_bias を入れたことに対応する。
#   ここが0.000と推定されていたら (1 | word) を置く根拠がデータに無い、
#   ということになる（実際、この項を入れる前の版ではそうなっていた）。


■ オッズ比・予測確率への変換：

In [ ]:
exp(fixef(model_glmm))              # expでログオッズを戻す（切片は「オッズ」、係数は「オッズ比」）
plogis(fixef(model_glmm)["(Intercept)"])  # voiced条件（基準水準）の予測確率。plogisはロジットの逆関数

# voiceless条件の予測確率は「係数を足してから」変換する。
# ★ここを間違えやすい。片方だけ変換した値を、変換していない値と足し引きしてはいけない。
plogis(sum(fixef(model_glmm)))

# --- 実行結果 ---
#        (Intercept) conditionvoiceless
#           2.590374           0.443243
# (Intercept)
#   0.7214775
# [1] 0.5344865
#
# 【出力の読み方】
# ・1つ目の出力のうち、**(Intercept)の2.590は「オッズ」であって「オッズ比」ではない**。
#   オッズ比は2つの群の比を指す言葉なので、基準水準1つだけのexp(0.9518)は
#   単なるオッズである。「voiced条件では、典型的と判断されるほうが
#   されないほうより2.59倍起こりやすい」と読む。
#   これに対しconditionvoicelessの0.443は正しく**オッズ比**であり、
#   「voicelessになるとそのオッズが0.443倍になる」＝**約56%減る**という意味。
#   **1が「変化なし」の基準**である点が、係数のときの0と違う。
# ・2つ目の 0.7215 は voiced条件で典型的と判断される**確率**である。
#   オッズ2.590を確率に直すと 2.590/(1+2.590) = 0.7215 になる。
#   オッズと確率は別物で、オッズ2.59は「2.59倍の確率」ではない。
# ・3つ目の 0.5344865 が voiceless条件の予測確率である。
#   **ログオッズの尺度で足してから plogis() で変換している**点が要点である。
#   0.9518 + (−0.8136) = 0.1382 → plogis(0.1382) ≒ 0.5345。
#   （表示は丸めた値なので、手計算とわずかにずれる）
#   ★よくある誤り: plogis(0.9518) − plogis(0.8136) のように、係数を
#   個別に変換してから引いたり、確率とオッズ比を混ぜて演算したりすること。
#   これは条件を正しく反映しない無意味な数字になる。**予測確率がほしいときは、
#   対象の条件のログオッズをすべて足し終えてから、最後に一度だけ変換する。**
# ・一方、**こうして正しく計算した予測確率どうしを引くことは問題ない。**
#   0.5345 − 0.7215 = −0.187 は確率差（リスク差）と呼ばれ、それ自体が
#   報告に値する効果量である。効果の大きさは、ログオッズでは−0.81、
#   オッズ比では0.443倍、確率では0.72→0.53（19ポイント減）と、目的に応じて
#   複数の尺度で表現できる。**論文では確率差かオッズ比のどちらかを、
#   何からの変化かを明記して報告する**——読者が実感できる尺度に直すところ
#   までが分析の仕事になる。
# ・★さらに注意すべき点: ロジスティック回帰の効果は**確率の尺度では一定でない**。
#   同じ−0.81でも、元の確率が0.5付近なら20ポイント動くが、0.95付近なら
#   数ポイントしか動かない。「この条件は正答率を19ポイント下げる」と
#   一般化して書くと誤りになる。**どの水準からの変化かを必ず添える。**
# ・★もう1点、GLMMに固有の注意：ここで計算した確率は、ランダム効果が
#   ちょうど0の（平均的な）話者・単語についての**条件付き確率**である。
#   実在するすべての話者・単語について確率を計算してから平均する
#   **母集団平均確率**とは、ロジットのような非線形リンクを挟む限り一般には
#   一致しない（Jensenの不等式）。母集団平均を報告したいときは
#   `marginaleffects`パッケージなどでのシミュレーションベースの周辺化を検討する。


■ ログオッズ → オッズ → 確率を1ステップずつ往復する：

上の変換を、有声条件の切片0.9518を例に1ステップずつ確かめる。ログオッズ→オッズ→確率と進み、確率0.72から逆にログオッズへ戻る。

In [ ]:
b0 <- fixef(model_glmm)["(Intercept)"]   # 切片（ログオッズ）
b0
odds <- exp(b0)                          # ログオッズ → オッズ
odds
odds / (1 + odds)                        # オッズ → 確率（plogis(b0) と同じ）
log(odds)                                # オッズ → ログオッズ（b0 に戻る）

qlogis(0.72)                             # 確率 → ログオッズ（plogisの逆関数）
exp(qlogis(0.72))                        # 確率 → オッズ
0.72 / (1 - 0.72)                        # 確率 → オッズ（定義どおりの計算）
plogis(qlogis(0.72))                     # 往復して0.72に戻る

# 条件の係数を足してから変換する手順も同じ往復で確かめられる
lp <- sum(fixef(model_glmm))             # 0.9518 + (-0.8136)（ログオッズ）
lp
exp(lp)                                  # オッズ
exp(b0) * exp(fixef(model_glmm)["conditionvoiceless"])  # オッズの尺度では掛け算
plogis(lp)                               # 確率
qlogis(0.5)                              # 確率0.5はログオッズ0

# --- 実行結果 ---
# (Intercept)
#   0.9518024            # b0
# (Intercept)
#    2.590374            # odds
# (Intercept)
#   0.7214775            # odds / (1 + odds)
# (Intercept)
#   0.9518024            # log(odds)
# [1] 0.9444616          # qlogis(0.72)
# [1] 2.571429           # exp(qlogis(0.72))
# [1] 2.571429           # 0.72 / (1 - 0.72)
# [1] 0.72               # plogis(qlogis(0.72))
# [1] 0.1381652          # lp
# [1] 1.148165           # exp(lp)
# (Intercept)
#    1.148165            # exp(b0) * exp(係数)
# [1] 0.5344865          # plogis(lp)
# [1] 0                  # qlogis(0.5)
#
# 【出力の読み方】
# ・ログオッズ0.9518を exp() に通すとオッズ2.590、odds/(1+odds) で確率0.7215。
#   plogis() はこの2ステップを1回で行う関数で、log(odds) で0.9518に戻る。
# ・逆向きに、確率0.72から qlogis() でログオッズ0.9445、exp() でオッズ2.571。
#   これは定義どおりの 0.72/(1-0.72) と一致する。確率0.72は「72%と28%」、
#   オッズ2.571は「72を28で割った比」であって、確率の2.571倍ではない。
# ・0.9445がモデルの切片0.9518（確率0.7215）とわずかに違うのは、0.72が
#   0.7215より少し低い確率だからである。
# ・「差がない」点は、ログオッズでは0、オッズでは1、確率では0.5。
# ・係数を足すのはログオッズの尺度、掛け算になるのはオッズの尺度。
#   0.1382 → オッズ1.148 → 確率0.5345 は、上のvoiceless条件の値と一致する。


■ 概念をつかむ：条件付き確率と周辺確率：

切片の plogis() はランダム効果が0の話者の確率（条件付き確率）で、話者全体を平均した確率（周辺確率）とは別の量である。切片0.9518と話者ランダム切片SD 0.6096を使い、仮想話者100万人を抽出して平均した値を、SDを3に大きくした場合と並べて比べる。

In [ ]:
# 前提: vot_data と model_glmm は直前のセルで作成済み。ここでは固定効果の切片と
# 話者ランダム切片SDを数値として置いて計算する（章の値: 0.9518, 0.6096）。
b0       <- 0.9518    # 有声条件の切片（ログオッズ）
sd_model <- 0.6096    # 話者ランダム切片のSD（章の推定値）
set.seed(1)
n_sim <- 1e6          # 仮想話者100万人

marginal_p <- function(b0, sd_u, n = n_sim) {
  u <- rnorm(n, mean = 0, sd = sd_u)   # 話者ごとのずれ（ログオッズ）
  mean(plogis(b0 + u))                 # 変換してから平均
}

cond_p <- plogis(b0)                   # (a) ランダム効果0の条件付き確率
res <- data.frame(
  scenario    = c("SD = 0.6096（章の推定値）", "SD = 3（大きなばらつき）"),
  conditional = round(c(cond_p, cond_p), 4),
  marginal    = round(c(marginal_p(b0, sd_model), marginal_p(b0, 3)), 4)
)
res$diff <- round(res$marginal - res$conditional, 4)
print(res)

# --- 実行結果 ---
#                    scenario conditional marginal    diff
# 1 SD = 0.6096（章の推定値）      0.7215   0.7068 -0.0147
# 2  SD = 3（大きなばらつき）      0.7215   0.6080 -0.1135
#
# 【出力の読み方】
# ・条件付き確率（ランダム効果0）はSDを変えても0.7215のまま動かない。
# ・周辺確率（仮想話者100万人の平均）は、SD 0.6096では0.7068（差は約1.5ポイント）、
#   SD 3では0.6080（差は約11ポイント）。ばらつきが大きいほど、条件付き確率と周辺確率の差は開く。
# ・単語のランダム切片は含めていないので、実際の差はもう少し大きくなる。
#   乱数を使うため、最後の桁は set.seed() と抽出数で変わる。


■ 3値以上のカテゴリ応答：多項ロジスティック回帰：

応答が3つ以上の名義カテゴリになる場面（日本語のピッチアクセント型の判定など）では、多項ロジスティック回帰（`nnet::multinom()`）を使う。

In [ ]:
install.packages("nnet", repos = "https://cloud.r-project.org")
library(nnet)

# 擬似データ: F0下降が語のどの位置で起きるか（fall_pos, 0=語頭付近, 1=語末付近）
# という連続的な音響手がかりから、4つのアクセント型を予測する。
set.seed(6)
accent_types <- c("平板型", "頭高型", "中高型", "尾高型")
d <- expand_grid(speaker = paste0("S", 1:12), word = 1:20) %>%
  mutate(
    true_type = sample(accent_types, n(), replace = TRUE, prob = c(0.35, 0.25, 0.20, 0.20)),
    fall_pos = case_when(
      true_type == "平板型" ~ rnorm(n(), 0.80, 0.22),
      true_type == "頭高型" ~ rnorm(n(), 0.18, 0.20),
      true_type == "中高型" ~ rnorm(n(), 0.45, 0.20),
      true_type == "尾高型" ~ rnorm(n(), 0.65, 0.20)
    ),
    fall_pos = pmin(pmax(fall_pos, 0), 1),
    true_type = factor(true_type, levels = accent_types)
  )

model_multi <- multinom(true_type ~ fall_pos, data = d)
summary(model_multi)

# 基準カテゴリ（因子の最初の水準="平板型"）に対する、他3水準のログオッズを表す
# 3組の係数が同時に推定される。

predict(model_multi, data.frame(fall_pos = c(0.1, 0.5, 0.9)), type = "probs")
# type="probs"を指定すると、係数の代わりに各カテゴリの予測確率が得られる。

cat("分類精度:", round(mean(predict(model_multi, d) == d$true_type), 3),
    " チャンスレベル:", round(max(table(d$true_type)) / nrow(d), 3), "\n")

# --- 実行結果（抜粋）---
# Coefficients:
#        (Intercept)   fall_pos
# 頭高型    7.374      -16.333
# 中高型    4.463       -8.487
# 尾高型    2.889       -4.933
#
#   平板型 頭高型 中高型 尾高型
# 1  0.003  0.864  0.103  0.030   (fall_pos = 0.1)
# 2  0.237  0.107  0.295  0.361   (fall_pos = 0.5)
# 3  0.797  0.001  0.033  0.169   (fall_pos = 0.9)
#
# 分類精度: 0.567  チャンスレベル: 0.342
#
# 【出力の読み方】
# ・頭高型の切片7.374は「fall_posが0のとき、平板型より頭高型のほうがログオッズで
#   7.374高い」、係数-16.333は「fall_posが1増えるごとにそのログオッズが16.333
#   下がる」という意味である。**基準カテゴリをどれにするかで係数の見た目は変わるが、
#   予測確率そのものは変わらない**（第4章のコーディングの選び方と同じ構造）。
# ・fall_pos=0.1（語頭付近で下降）ならほぼ確実に頭高型（86.4%）、fall_pos=0.9
#   （語末付近で下降）なら平板型が最有力（79.7%）になる。fall_pos=0.5（中間）
#   では複数の型が競合し、どれか1つに強く偏らない。
# ・分類精度56.7%はチャンスレベル（最頻クラス34.2%）を上回るが、100%には遠い。
#   fall_posという1つの手がかりだけでは型どうしの重なりが大きいためであり、
#   これは「この手がかり・このモデルでの達成精度」であって理論的な上限ではない
#   （第14章参照）。
# ・★話者・単語のランダム効果を含めたい場合は`brms::brm(family = categorical())`
#   を使う。`nnet::multinom()`はランダム効果を扱えないため、交差デザインの
#   データでは擬似反復（第3章・第5章）の問題が残る。


■ 二山に分かれる連続量：ハードルモデル：

第14章で扱った「前voicingを使うかどうか」というVOTの二峰性を、実際にモデル化する。前voicingの有無（離散）と、使った場合の長さ（連続）は性質の異なる2つの過程なので、1本の線形モデルに混ぜてはいけない。

**変数を2つに分けて定義することが出発点になる。** `uses_prevoice`（前voicingを使ったかどうか、二値）と`prevoice_duration`（使った場合の声帯振動の持続時間、**常に正の値**）を別々の列として持ち、符号付きの`VOT`（前voicingを使った行では`VOT = -prevoice_duration`となる、破裂と声帯振動開始の前後関係を表す量）とは区別する。VOTの符号（負）をそのまま引き継いだ変数を持続時間のモデルに使うと、符号を読み違えやすい。

In [ ]:
set.seed(66)
d <- expand_grid(speaker = paste0("S", 1:12), word = 1:20) %>%
  mutate(
    condition = rep(c("normal", "fast"), length.out = n()),
    sp_re = rep(rnorm(12, 0, 0.5), each = 20)[as.integer(factor(speaker))],
    p_prevoice = plogis(0.4 + ifelse(condition == "fast", -1.0, 0) + sp_re),
    uses_prevoice = rbinom(n(), 1, p_prevoice),
    # prevoice_duration: 前voicingを使った場合の声帯振動の持続時間（常に正の値）
    prevoice_duration = ifelse(uses_prevoice == 1, rgamma(n(), shape = 9, rate = 0.15), NA),
    # VOT: 音声学的な符号付きの測定値（前voicingを使った行では負、それ以外は正）
    VOT = ifelse(uses_prevoice == 1, -prevoice_duration, rnorm(n(), 5, 8))
  )

cat("=== 誤り: 全VOTをそのまま線形混合モデルにかける ===\n")
print(coef(summary(lmer(VOT ~ condition + (1 | speaker), data = d))))

cat("\n=== 正しい分析: 二部モデル ===\n")
cat("第1段階（使うかどうか）:\n")
print(coef(summary(glmer(uses_prevoice ~ condition + (1 | speaker),
                          data = d, family = binomial))))

cat("\n第2段階（使った場合の持続時間、prevoice_duration・正の値）:\n")
print(coef(summary(lmer(prevoice_duration ~ condition + (1 | speaker),
                         data = filter(d, uses_prevoice == 1)))))

# --- 実行結果（抜粋）---
# 誤り（全VOTをそのまま）:
#                   Estimate Std. Error    t value
# (Intercept)     -39.268      3.079    -12.753
# conditionnormal  -9.409      4.332     -2.172
#
# 第1段階（使うかどうか、二項GLMM）:
#                  Estimate Std. Error z value  Pr(>|z|)
# (Intercept)        0.745      0.214   3.480    0.0005
# conditionnormal    0.614      0.300   2.046    0.0408
#
# 第2段階（使った場合の持続時間、前voicingが起きたトークンだけのLMM）:
# boundary (singular) fit: see help('isSingular')
#                   Estimate Std. Error t value
# (Intercept)        61.156      2.099   29.131
# conditionnormal     1.036      2.857    0.363
#
# 【出力の読み方】
# ・誤った分析（全VOTをそのまま）は−9.409msという一見それらしい係数を返すが、
#   これが「前voicingの使用率が変わった効果」なのか「使った場合の長さが変わった
#   効果」なのか、この1本のモデルからは区別できない。
# ・二部モデルに分けると、条件は**使うかどうかには有意な効果を持つ**
#   （p = 0.041、通常の速さのほうが前voicingを使いやすい）が、**使った場合の
#   持続時間には効果を持たない**（p = 0.72）ことが分かる。1本の線形モデルの
#   −9.4msは、この2つの異なる効果が混ざった、単独では解釈できない数値だった。
# ・prevoice_durationはVOTの符号を引き継がず、常に正の値として定義してある
#   ので、第2段階の係数（61.156、1.036）はそのまま「ms単位の持続時間」として
#   読める。もしVOTの符号をそのまま引き継いだ変数を使っていたら、係数は
#   負（-61.156、-1.036）になり、「持続時間が負」という読み違えを招きかねない。
# ・第2段階のモデルは`boundary (singular) fit`の警告を出す——フィルタ後の
#   データ数が少なく、話者のランダム切片の分散が0に潰れているためである。
#   第5章5.6節の基準に従い、ランダム効果の構造を単純化するか、収束の警告として
#   明記して報告する。
# ・どちらの過程に効果があるかは研究上まったく異なる意味を持つ。離散的な
#   カテゴリ選択の変化と、連続的な調音のタイミング調整を、1本の線形モデルで
#   済ませると区別そのものが失われる。


■ 演習：

1. `family = binomial` を `family = binomial(link = "probit")` に変えて結果がどう変わるか比較せよ。
2. 応答をカウントデータ（例: 1発話あたりの誤り数）に見立て、`family = poisson` のGLMMを組んでみよ。

■ ★よくある誤り：割合に集計してからt検定・ANOVAをかける：

二値データの分析で最も多い誤りが、**話者ごとに正答率（割合）を計算してからt検定や分散分析にかける**ことである。Jaeger (2008) はこの手法を明示的に批判し、ロジットのGLMMを推奨した。何が失われるのかを実際に比べてみる。

In [ ]:
# ---- 誤った分析: 話者ごとの割合に集計してからt検定 ----
agg <- vot_data %>%
  group_by(speaker, condition) %>%
  summarise(p = mean(typical), .groups = "drop") %>%
  pivot_wider(names_from = condition, values_from = p)

tt <- t.test(agg$voiced, agg$voiceless, paired = TRUE)
cat("割合の対応ありt検定 : t =", round(tt$statistic, 3),
    " df =", tt$parameter, " p =", round(tt$p.value, 4), "\n")

# arcsine平方根変換を挟む流儀（かつて広く使われた）も試す
tt2 <- t.test(asin(sqrt(agg$voiced)), asin(sqrt(agg$voiceless)), paired = TRUE)
cat("arcsine変換後のt検定: t =", round(tt2$statistic, 3),
    " df =", tt2$parameter, " p =", round(tt2$p.value, 4), "\n")

# ---- 正しい分析: GLMM ----
co <- summary(model_glmm)$coefficients
cat("GLMM                : z =", round(co[2, 3], 3),
    " p =", round(co[2, 4], 4), "\n")

# ---- 集計によって何が捨てられたかを数える ----
cat("\n集計前の観測数        :", nrow(vot_data), "\n")
cat("集計後の観測数        :", nrow(agg) * 2, "\n")
cat("単語ごとの正答率のSD  :", round(sd(tapply(vot_data$typical, vot_data$word, mean)), 3), "\n")

# --- 実行結果 ---
# 割合の対応ありt検定 : t = 2.375  df = 9  p = 0.0415
# arcsine変換後のt検定: t = 2.395  df = 9  p = 0.0402
# GLMM                : z = -2.194  p = 0.0282
#
# 集計前の観測数        : 200
# 集計後の観測数        : 20
# 単語ごとの正答率のSD  : 0.195
#
# 【出力の読み方】
# ・**このデータでは3つの方法が同じ結論（有意）に達している。**
#   誠実に書けば、集計してt検定をかけたからといって必ず結論が変わるわけではない。
#   しかし結論が同じであることと、手法として妥当であることは別問題である。
# ・第一の問題は、**集計によって200件の観測が20件に減っている**ことである。
#   自由度は9まで落ち、1試行ごとの情報——どのVOT値のときに典型と判断されたか——は
#   完全に失われる。第6章の生成コードで入れた「典型値から離れるほど非典型と
#   判断される」という試行レベルの効果は、集計した時点で分析不可能になる。
# ・第二の問題がより深刻である。**集計した表には単語の情報が残らない。**
#   単語ごとの正答率は標準偏差0.195もばらついているのに、
#   話者ごとに平均してしまうとこのばらつきは誤差に紛れ込む。
#   結果として (1 | word) を置くことができなくなる。**これは項目を
#   固定効果として扱うのと同じではない**——固定効果なら各単語に固有の
#   係数を推定してその効果を残せるが、話者ごとの集計は単語という要因の
#   存在そのものを表から消してしまう、より情報量の少ない扱いである。
#   Clark (1973) 以来「言語をランダム効果として扱わないと、その言語材料に
#   固有の結果を一般化してしまう」と繰り返し指摘されてきた問題であり、
#   Jaeger (2008) の批判の核心もここにある。
# ・第三に、割合という尺度そのものが線形モデルに向かない。割合は0〜1に
#   閉じているのに、線形モデルは予測値が0を下回ったり1を超えたりすることを
#   許してしまう。また分散が平均に依存する（0.5付近で最大、両端で最小）ため、
#   等分散性の仮定も満たさない。arcsine変換はこの分散の問題に対処するために
#   考案されたが、**上の出力のとおり結果はほとんど変わらず**、
#   一方で係数の解釈可能性を失う（arcsine変換後の値は意味を持たない）。
# ・★正答率が0%または100%の話者・条件があると、この問題はさらに悪化する。
#   割合が0や1に張り付くとarcsine変換は端で潰れ、t検定の前提から大きく外れる。
#   GLMMは試行を1件ずつ扱うので、この問題自体が起きない。
# ・結論として、**二値の応答は集計せずGLMMで扱う**のが既定の選択である。
#   集計が正当化されるのは、試行数がどの条件でも完全に等しく、
#   項目差が無視できると示せる場合に限られる。


■ 演習の解答：

1問目はリンク関数を変える練習、2問目は応答の種類に応じて分布族を選び分ける練習である。

In [ ]:
# 1. probitリンクに変えて比較する。
#    logit と probit は、ロジスティック分布と正規分布のどちらを仮定するかの違い。
model_probit <- glmer(
  typical ~ condition + (1 | speaker) + (1 | word),
  family = binomial(link = "probit"),
  data = vot_data
)

cat("logit  の係数:", round(fixef(model_glmm)["conditionvoiceless"], 4), "\n")
cat("probit の係数:", round(fixef(model_probit)["conditionvoiceless"], 4), "\n")
cat("係数の比      :", round(fixef(model_glmm)["conditionvoiceless"] /
                              fixef(model_probit)["conditionvoiceless"], 3), "\n")
cat("logit  のAIC :", round(AIC(model_glmm), 2), "\n")
cat("probit のAIC :", round(AIC(model_probit), 2), "\n")

# 予測確率はほぼ同じになる（リンク関数が違っても当てはめる曲線は似ている）
cat("voiced条件の予測確率 logit :",
    round(plogis(fixef(model_glmm)["(Intercept)"]), 4), "\n")
cat("voiced条件の予測確率 probit:",
    round(pnorm(fixef(model_probit)["(Intercept)"]), 4), "\n")

# 2. カウントデータをポアソンGLMMで扱う。
#    「1発話あたりの言い直し回数」を擬似的に作る。カウントは0以上の整数で、
#    上限が無い点が二値データと違う。
set.seed(60)
vot_count <- vot_data %>%
  mutate(
    # 無声条件のほうがやや言い直しが多いと仮定（log尺度で+0.4）
    lambda = exp(0.2 + if_else(condition == "voiceless", 0.4, 0) + speaker_bias),
    n_repair = rpois(n(), lambda = lambda)
  )

table(vot_count$n_repair)  # 0が多く右に裾を引く、典型的なカウント分布

model_pois <- glmer(
  n_repair ~ condition + (1 | speaker) + (1 | word),
  family = poisson,
  data = vot_count
)
summary(model_pois)

# ★過分散（overdispersion）の確認。ポアソンは「平均=分散」を仮定するので、
#   実データではしばしば分散が大きすぎて仮定が破れる。
pearson_chi <- sum(residuals(model_pois, type = "pearson")^2)
cat("\n過分散の指標 (1に近ければ良い):",
    round(pearson_chi / df.residual(model_pois), 3), "\n")

# --- 実行結果（抜粋）---
# logit  の係数: -0.8136
# probit の係数: -0.4978
# 係数の比      : 1.634
# logit  のAIC : 262.65
# probit のAIC : 262.66
# voiced条件の予測確率 logit : 0.7215
# voiced条件の予測確率 probit: 0.7203
#
#  0  1  2  3  4  5  6  7  9 10
# 40 56 44 25 16  8  4  4  2  1
#
# Random effects:
#  Groups  Name        Variance Std.Dev.
#  word    (Intercept) 0.0339   0.1841
#  speaker (Intercept) 0.3282   0.5729
#
# Fixed effects:
#                    Estimate Std. Error z value Pr(>|z|)
# (Intercept)          0.3174     0.2081   1.525  0.12726
# conditionvoiceless   0.3857     0.1303   2.960  0.00308 **
#
# 過分散の指標 (1に近ければ良い): 0.817
#
# 【出力の読み方】
# ・1問目: logitの係数 −0.8136 に対し、probitは −0.4978 と**小さく出る**。
#   比は1.634で、理論値の約1.6〜1.7に一致する。これはリンク関数が仮定する
#   分布の分散が違うためで、**モデルの当てはまりが違うのではなく尺度が違う**。
#   AICは262.65対262.66とほぼ同一、予測確率も0.7215対0.7203でほぼ同じである。
# ・したがって「probitに変えたら係数が小さくなった」ことに意味は無い。
#   **logitとprobitの係数を直接比べてはいけない。** 比べるなら予測確率で比べる。
# ・使い分けの目安: logitはオッズ比として解釈できるため、報告のしやすさで有利。
#   probitは背後に正規分布の潜在変数を仮定するので、信号検出理論（d'）や
#   閾値モデルと接続したいときに自然である。**音声知覚の同定曲線では
#   probitが選ばれることも多い**が、当てはまりで選ぶ問題ではない。
# ・2問目: 度数分布 0が40件、1が56件、…、10が1件という形は、
#   **0付近に集中して右に長い裾を引く**典型的なカウント分布である。
#   この形のデータに線形モデルを当てると、予測値が負になったり、
#   残差の正規性が破れたりする。だからポアソン分布を使う。
# ・係数 0.3857 は**log尺度**である。exp(0.3857) = 1.471 なので
#   「無声条件では言い直しの回数が約1.47倍になる」と読む。
#   生成時にlog尺度で+0.4を入れたので、正しく回収できている。
#   ★二項GLMMのときと同じく、**係数をそのまま「回数」として読まない**。
# ・過分散の指標 0.817 は1に近く、問題なしと判断できる。
#   ポアソン分布は「平均=分散」を仮定するが、実データでは分散のほうが
#   大きくなること（過分散）が多い。**この値が1.5や2を超えたら
#   ポアソンの仮定は破れている**ので、負の二項分布（`glmmTMB` の
#   `family = nbinom2`）に切り替えるか、観測ごとのランダム効果を足す。
# ・★過分散を放置すると標準誤差が過小になり、**有意でないものが有意に見える**。
#   カウントデータでポアソンを使ったら、この指標の確認は必須である。
